In [1]:
import os
import random
import numpy as np
import pandas as pd
import torch

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)

In [2]:
import torch
import transformers
from transformers import AutoTokenizer, AutoModelForSequenceClassification

print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("cuda:", torch.cuda.is_available())

if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

torch: 2.5.1+cu121
transformers: 4.45.2
cuda: True
NVIDIA GeForce RTX 4060 Laptop GPU


In [3]:
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

PyTorch version: 2.5.1+cu121
CUDA available: True
GPU: NVIDIA GeForce RTX 4060 Laptop GPU
CUDA version: 12.1


device(type='cuda')

In [4]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [5]:
train_df = pd.read_csv("../data/processed/train_soft_labels.csv")
val_df = pd.read_csv("../data/processed/validation_soft_labels.csv")
test_df = pd.read_csv("../data/processed/test_soft_labels.csv")

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

Train: (13832, 23)
Validation: (2964, 23)
Test: (2965, 23)


In [6]:
text_col = "text_clean"

label_cols = [
    "hatespeech_0_prob",
    "hatespeech_1_prob",
    "hatespeech_2_prob"
]

num_labels = len(label_cols)

train_df[[text_col] + label_cols].head()

,text_clean,hatespeech_0_prob,hatespeech_1_prob,hatespeech_2_prob
0,First off you look cool as fuck! Anyway if we ...,1.000000,0.0,0.000000
1,"They'll come back in your plan, also. Plus we ...",0.333333,0.0,0.666667
2,She ugly anyways,0.500000,0.0,0.500000
3,Do they realize that a random Japanese person ...,1.000000,0.0,0.000000
4,Won't happen. Birth rates are down. Women want...,1.000000,0.0,0.000000


In [7]:
for name, df in [("train", train_df), ("validation", val_df), ("test", test_df)]:
    sums = df[label_cols].sum(axis=1)
    print(name)
    print("Min label sum:", sums.min())
    print("Max label sum:", sums.max())
    print("Mean label sum:", sums.mean())
    print()

train
Min label sum: 0.9999999999999998
Max label sum: 1.0
Mean label sum: 1.0

validation
Min label sum: 0.9999999999999999
Max label sum: 1.0
Mean label sum: 1.0

test
Min label sum: 0.9999999999999999
Max label sum: 1.0
Mean label sum: 1.0



In [8]:
train_dataset = Dataset.from_pandas(train_df[[text_col] + label_cols].reset_index(drop=True))
val_dataset = Dataset.from_pandas(val_df[[text_col] + label_cols].reset_index(drop=True))
test_dataset = Dataset.from_pandas(test_df[[text_col] + label_cols].reset_index(drop=True))

train_dataset

Dataset({
    features: ['text_clean', 'hatespeech_0_prob', 'hatespeech_1_prob', 'hatespeech_2_prob'],
    num_rows: 13832
})

In [9]:
model_name = "roberta-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)

In [10]:
MAX_LENGTH = 128

def tokenize(batch):
    return tokenizer(
        batch[text_col],
        truncation=True,
        max_length=MAX_LENGTH
    )

train_dataset = train_dataset.map(tokenize, batched=True)
val_dataset = val_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

Map:   0%|          | 0/13832 [00:00<?, ? examples/s]

Map:   0%|          | 0/2964 [00:00<?, ? examples/s]

Map:   0%|          | 0/2965 [00:00<?, ? examples/s]

In [11]:
def add_soft_labels(batch):
    labels = []

    for i in range(len(batch[label_cols[0]])):
        labels.append([
            float(batch["hatespeech_0_prob"][i]),
            float(batch["hatespeech_1_prob"][i]),
            float(batch["hatespeech_2_prob"][i])
        ])

    batch["labels"] = labels
    return batch

train_dataset = train_dataset.map(add_soft_labels, batched=True)
val_dataset = val_dataset.map(add_soft_labels, batched=True)
test_dataset = test_dataset.map(add_soft_labels, batched=True)

Map:   0%|          | 0/13832 [00:00<?, ? examples/s]

Map:   0%|          | 0/2964 [00:00<?, ? examples/s]

Map:   0%|          | 0/2965 [00:00<?, ? examples/s]

In [12]:
columns_to_keep = [
    "input_ids",
    "attention_mask",
    "labels"
]

train_dataset.set_format(type="torch", columns=columns_to_keep)
val_dataset.set_format(type="torch", columns=columns_to_keep)
test_dataset.set_format(type="torch", columns=columns_to_keep)

In [13]:
data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)

In [14]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels
)

model.to(device)

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
         

In [15]:
class SoftLabelTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels").float()

        outputs = model(**inputs)
        logits = outputs.logits

        log_probs = torch.nn.functional.log_softmax(logits, dim=-1)

        loss = torch.nn.functional.kl_div(
            log_probs,
            labels,
            reduction="batchmean"
        )

        return (loss, outputs) if return_outputs else loss

In [16]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred

    logits = torch.tensor(logits)
    labels = np.array(labels)

    probs = torch.nn.functional.softmax(logits, dim=-1).numpy()

    eps = 1e-12

    kl = np.sum(
        labels * (np.log(labels + eps) - np.log(probs + eps)),
        axis=1
    ).mean()

    m = 0.5 * (labels + probs)

    js = 0.5 * (
        np.sum(labels * (np.log(labels + eps) - np.log(m + eps)), axis=1)
        +
        np.sum(probs * (np.log(probs + eps) - np.log(m + eps)), axis=1)
    ).mean()

    pred_hard = probs.argmax(axis=1)
    gold_hard = labels.argmax(axis=1)

    hard_accuracy = (pred_hard == gold_hard).mean()

    pred_entropy = -np.sum(probs * np.log2(probs + eps), axis=1)
    gold_entropy = -np.sum(labels * np.log2(labels + eps), axis=1)

    entropy_corr = np.corrcoef(pred_entropy, gold_entropy)[0, 1]

    return {
        "kl_divergence": float(kl),
        "js_divergence": float(js),
        "hard_accuracy": float(hard_accuracy),
        "entropy_correlation": float(entropy_corr)
    }

In [17]:
training_args = TrainingArguments(
    output_dir="../models/roberta_soft_hatespeech_baseline_1",

    eval_strategy="epoch",
    save_strategy="epoch",

    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,

    num_train_epochs=15,
    weight_decay=0.01,

    logging_steps=100,
    load_best_model_at_end=True,

    metric_for_best_model="js_divergence",
    greater_is_better=False,

    fp16=torch.cuda.is_available(),

    report_to="none",
    seed=SEED
)

In [18]:
trainer = SoftLabelTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

/home/shayan/Distributional-Hate-Speech-Prediction/.venv/lib/python3.12/site-packages/accelerate/accelerator.py:494: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


In [19]:
torch.cuda.empty_cache()
trainer.train()

Epoch,Training Loss,Validation Loss,Kl Divergence,Js Divergence,Hard Accuracy,Entropy Correlation
1,0.535200,0.516776,0.516776,0.141195,0.767544,0.247056
2,0.486500,0.550775,0.550775,0.130922,0.768219,0.259017
3,0.420900,0.558947,0.558947,0.135647,0.755061,0.248590
4,0.344100,0.644738,0.644738,0.144603,0.725034,0.234241
5,0.308700,0.733603,0.733603,0.135012,0.759447,0.223685
6,0.280900,0.817297,0.817297,0.140671,0.745951,0.229967
7,0.215200,0.933852,0.933852,0.158840,0.721660,0.229164
8,0.181900,0.890339,0.890339,0.148863,0.732119,0.211154
9,0.171400,0.950424,0.950424,0.144646,0.752362,0.212585
10,0.137100,1.078415,1.078415,0.148787,0.742240,0.208223


TrainOutput(global_step=25935, training_loss=0.22970499090544402, metrics={'train_runtime': 2434.2476, 'train_samples_per_second': 85.234, 'train_steps_per_second': 10.654, 'total_flos': 8324411088108528.0, 'train_loss': 0.22970499090544402, 'epoch': 15.0})

In [20]:
val_results = trainer.evaluate(val_dataset)
val_results

{'eval_loss': 0.5507746338844299,
 'eval_kl_divergence': 0.5507746934890747,
 'eval_js_divergence': 0.13092166185379028,
 'eval_hard_accuracy': 0.7682186234817814,
 'eval_entropy_correlation': 0.2590165476810119,
 'eval_runtime': 3.7495,
 'eval_samples_per_second': 790.507,
 'eval_steps_per_second': 49.607,
 'epoch': 15.0}

In [21]:
test_results = trainer.evaluate(test_dataset)
test_results

{'eval_loss': 0.5855312347412109,
 'eval_kl_divergence': 0.5855312347412109,
 'eval_js_divergence': 0.13706298172473907,
 'eval_hard_accuracy': 0.7514333895446881,
 'eval_entropy_correlation': 0.2375989333117849,
 'eval_runtime': 3.8119,
 'eval_samples_per_second': 777.836,
 'eval_steps_per_second': 48.795,
 'epoch': 15.0}

In [22]:
model_save_path = "../models/roberta_soft_hatespeech_baseline_1/final_model"

trainer.save_model(model_save_path)
tokenizer.save_pretrained(model_save_path)

print("Saved model to:", model_save_path)

Saved model to: ../models/roberta_soft_hatespeech_baseline_1/final_model


In [23]:
os.makedirs("../results/tables", exist_ok=True)

results_path = "../results/tables/roberta_soft_hatespeech_baseline_1_results.txt"

with open(results_path, "w", encoding="utf-8") as f:
    f.write("ROBERTA SOFT-LABEL HATESPEECH BASELINE RESULTS\n")
    f.write("=" * 70 + "\n\n")

    f.write("Model:\n")
    f.write(f"- {model_name}\n\n")

    f.write("CUDA:\n")
    f.write(f"- CUDA available: {torch.cuda.is_available()}\n")
    if torch.cuda.is_available():
        f.write(f"- GPU: {torch.cuda.get_device_name(0)}\n")
        f.write(f"- CUDA version: {torch.version.cuda}\n")
    f.write("\n")

    f.write("Training setup:\n")
    f.write(f"- Max length: {MAX_LENGTH}\n")
    f.write(f"- Train batch size: {training_args.per_device_train_batch_size}\n")
    f.write(f"- Eval batch size: {training_args.per_device_eval_batch_size}\n")
    f.write(f"- Epochs: {training_args.num_train_epochs}\n")
    f.write(f"- Learning rate: {training_args.learning_rate}\n")
    f.write(f"- FP16: {training_args.fp16}\n\n")

    f.write("Validation results:\n")
    for key, value in val_results.items():
        f.write(f"{key}: {value}\n")

    f.write("\nTest results:\n")
    for key, value in test_results.items():
        f.write(f"{key}: {value}\n")

print("Saved:", results_path)

Saved: ../results/tables/roberta_soft_hatespeech_baseline_1_results.txt


In [ ]:
import json
from datetime import datetime

os.makedirs("../results/tables", exist_ok=True)

results_path = "../results/tables/roberta_soft_hatespeech_baseline_1_results.txt"
history_path = "../results/tables/roberta_soft_hatespeech_baseline_1_training_history.csv"

# Convert Trainer log history to a dataframe
history_df = pd.DataFrame(trainer.state.log_history)

# Save full training history separately as CSV
history_df.to_csv(history_path, index=False)

# Extract useful training/evaluation logs
train_logs = history_df[history_df["loss"].notna()] if "loss" in history_df.columns else pd.DataFrame()
eval_logs = history_df[history_df["eval_loss"].notna()] if "eval_loss" in history_df.columns else pd.DataFrame()

with open(results_path, "w", encoding="utf-8") as f:
    f.write("ROBERTA SOFT-LABEL HATESPEECH BASELINE 1 RESULTS\n")
    f.write("=" * 80 + "\n\n")

    f.write("1. RUN INFORMATION\n")
    f.write("-" * 80 + "\n")
    f.write(f"Run timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write(f"Model name: {model_name}\n")
    f.write(f"Output directory: {training_args.output_dir}\n")
    f.write(f"Best model checkpoint: {trainer.state.best_model_checkpoint}\n")
    f.write(f"Best validation metric: {trainer.state.best_metric}\n")
    f.write(f"Global training steps: {trainer.state.global_step}\n\n")

    f.write("2. CUDA / HARDWARE\n")
    f.write("-" * 80 + "\n")
    f.write(f"CUDA available: {torch.cuda.is_available()}\n")
    if torch.cuda.is_available():
        f.write(f"GPU: {torch.cuda.get_device_name(0)}\n")
        f.write(f"CUDA version: {torch.version.cuda}\n")
    f.write("\n")

    f.write("3. DATASET SIZES\n")
    f.write("-" * 80 + "\n")
    f.write(f"Train rows: {len(train_df)}\n")
    f.write(f"Validation rows: {len(val_df)}\n")
    f.write(f"Test rows: {len(test_df)}\n\n")

    f.write("4. TARGET COLUMNS\n")
    f.write("-" * 80 + "\n")
    for col in label_cols:
        f.write(f"- {col}\n")
    f.write("\n")

    f.write("5. TRAINING SETUP\n")
    f.write("-" * 80 + "\n")
    f.write(f"Max sequence length: {MAX_LENGTH}\n")
    f.write(f"Train batch size: {training_args.per_device_train_batch_size}\n")
    f.write(f"Eval batch size: {training_args.per_device_eval_batch_size}\n")
    f.write(f"Epochs: {training_args.num_train_epochs}\n")
    f.write(f"Learning rate: {training_args.learning_rate}\n")
    f.write(f"Weight decay: {training_args.weight_decay}\n")
    f.write(f"FP16: {training_args.fp16}\n")
    f.write(f"Metric for best model: {training_args.metric_for_best_model}\n")
    f.write(f"Greater is better: {training_args.greater_is_better}\n\n")

    f.write("6. FINAL VALIDATION RESULTS\n")
    f.write("-" * 80 + "\n")
    for key, value in val_results.items():
        f.write(f"{key}: {value}\n")
    f.write("\n")

    f.write("7. FINAL TEST RESULTS\n")
    f.write("-" * 80 + "\n")
    for key, value in test_results.items():
        f.write(f"{key}: {value}\n")
    f.write("\n")

    f.write("8. TRAINING HISTORY SUMMARY\n")
    f.write("-" * 80 + "\n")
    f.write(f"Full training history saved to: {history_path}\n")
    f.write(f"Total log entries: {len(history_df)}\n")
    f.write(f"Training loss log entries: {len(train_logs)}\n")
    f.write(f"Evaluation log entries: {len(eval_logs)}\n\n")

    if not train_logs.empty:
        f.write("Training loss summary:\n")
        f.write(train_logs["loss"].describe().to_string())
        f.write("\n\n")

        f.write("First 5 training loss logs:\n")
        f.write(train_logs[["epoch", "step", "loss"]].head().to_string(index=False))
        f.write("\n\n")

        f.write("Last 5 training loss logs:\n")
        f.write(train_logs[["epoch", "step", "loss"]].tail().to_string(index=False))
        f.write("\n\n")

    if not eval_logs.empty:
        metric_cols = [
            col for col in [
                "epoch",
                "step",
                "eval_loss",
                "eval_kl_divergence",
                "eval_js_divergence",
                "eval_hard_accuracy",
                "eval_entropy_correlation"
            ]
            if col in eval_logs.columns
        ]

        f.write("Evaluation history by epoch:\n")
        f.write(eval_logs[metric_cols].to_string(index=False))
        f.write("\n\n")

        if "eval_js_divergence" in eval_logs.columns:
            best_js_row = eval_logs.loc[eval_logs["eval_js_divergence"].idxmin()]
            f.write("Best validation JS divergence epoch:\n")
            f.write(best_js_row[metric_cols].to_string())
            f.write("\n\n")

        if "eval_kl_divergence" in eval_logs.columns:
            best_kl_row = eval_logs.loc[eval_logs["eval_kl_divergence"].idxmin()]
            f.write("Best validation KL divergence epoch:\n")
            f.write(best_kl_row[metric_cols].to_string())
            f.write("\n\n")

        if "eval_entropy_correlation" in eval_logs.columns:
            best_entropy_row = eval_logs.loc[eval_logs["eval_entropy_correlation"].idxmax()]
            f.write("Best validation entropy correlation epoch:\n")
            f.write(best_entropy_row[metric_cols].to_string())
            f.write("\n\n")


print("Saved result report:", results_path)
print("Saved training history:", history_path)

Saved result report: ../results/tables/roberta_soft_hatespeech_baseline_1_results.txt
Saved training history: ../results/tables/roberta_soft_hatespeech_baseline_1_training_history.csv
